# PR. 1 - Expectation Decider

Mathematics & Advanced Statistics

This notebook analyzes probability patterns in a dataset of 200 students.

## 1. Dataset Generation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

n = 200

study_hours = np.random.randint(2, 21, n)
attendance = np.random.randint(40, 101, n)

group_discussion = np.random.choice(
    ["Yes", "No"], n, p=[0.6, 0.4]
)

previous_test_score = np.clip(
    study_hours * 2.5 +
    attendance * 0.45 +
    np.random.normal(0, 10, n),
    20, 100
).round().astype(int)

score = (
    study_hours * 2 +
    attendance * 0.3 +
    previous_test_score * 0.5 +
    (group_discussion == "Yes") * 8
)

probability = np.clip((score - 50) / 80, 0.05, 0.95)

final_exam_pass = np.where(
    np.random.rand(n) < probability,
    "Pass",
    "Fail"
)

df = pd.DataFrame({
    "study_hours": study_hours,
    "attendance": attendance,
    "group_discussion": group_discussion,
    "previous_test_score": previous_test_score,
    "final_exam_pass": final_exam_pass
})

df.to_csv("expectation_decider_dataset.csv", index=False)

df.head()

## 2. Understanding the Basics

**Probability** is the measure of how likely an event is to occur.

The basic formula is:

**P(E) = Favorable outcomes / Total outcomes**

Examples from the dataset:

1. Probability that a randomly selected student passes the exam.
2. Probability that a student has attendance greater than 80%.
3. Probability that a student participates in group discussion and passes the exam.

Important terms include experiment, outcome, sample space, event, favorable outcome, complementary event and conditional probability.

In [ ]:
total = len(df)

pass_count = (df["final_exam_pass"] == "Pass").sum()
high_attendance = (df["attendance"] > 80).sum()
group_pass = ((df["group_discussion"] == "Yes") &
              (df["final_exam_pass"] == "Pass")).sum()

prob_pass = pass_count / total
prob_high_attendance = high_attendance / total
prob_group_pass = group_pass / total

print("P(Pass) =", round(prob_pass, 4))
print("P(Attendance > 80%) =", round(prob_high_attendance, 4))
print("P(Group Discussion and Pass) =", round(prob_group_pass, 4))

## 3. Types of Probability

### Empirical Probability

Empirical probability is calculated from observed data.

**P(E) = Number of times event occurs / Total observations**

### Theoretical Probability

For this project, a simple theoretical example is selecting a final-exam outcome from the two possible outcomes, Pass and Fail, assuming both are equally likely.

**P(Pass) = 1 / 2 = 0.5**

The empirical value comes from the actual dataset, while the theoretical example is based on an equal-likelihood assumption.

In [ ]:
empirical_pass = pass_count / total
theoretical_pass = 1 / 2

print("Empirical probability of Pass =", round(empirical_pass, 4))
print("Theoretical probability of Pass =", theoretical_pass)

## 4. Random Variable and Probability Distribution

Let **X** be the number of students who pass when 3 students are randomly selected.

Therefore:

**X = 0, 1, 2, 3**

Using the pass probability from the dataset, X follows a binomial distribution.

**P(X = x) = C(n,x) p^x (1-p)^(n-x)**

For a binomial distribution:

**Mean = np**

**Variance = np(1-p)**

In [ ]:
from math import comb

p = prob_pass
n_students = 3

distribution = []

for x in range(4):
    px = comb(n_students, x) * p**x * (1-p)**(n_students-x)
    distribution.append([x, px])

distribution_df = pd.DataFrame(
    distribution,
    columns=["X", "P(X)"]
)

mean = n_students * p
variance = n_students * p * (1 - p)

print(distribution_df)
print("\nMean =", round(mean, 4))
print("Variance =", round(variance, 4))

## 5. Venn Diagram

Let:

**A = Students studying more than 10 hours/week**

**B = Students attending more than 80% of classes**

The overlap A ∩ B represents students satisfying both conditions.

In [ ]:
A = df["study_hours"] > 10
B = df["attendance"] > 80

a_only = (A & ~B).sum()
b_only = (~A & B).sum()
both = (A & B).sum()
neither = (~A & ~B).sum()

print("Study > 10 hours only:", a_only)
print("Attendance > 80% only:", b_only)
print("Both conditions:", both)
print("Neither condition:", neither)

In [ ]:
from matplotlib.patches import Circle

fig, ax = plt.subplots(figsize=(7, 5))

circle1 = Circle((0.42, 0.5), 0.25, fill=False, linewidth=2)
circle2 = Circle((0.58, 0.5), 0.25, fill=False, linewidth=2)

ax.add_patch(circle1)
ax.add_patch(circle2)

ax.text(0.32, 0.5, str(a_only), ha="center", va="center", fontsize=14)
ax.text(0.50, 0.5, str(both), ha="center", va="center", fontsize=14)
ax.text(0.68, 0.5, str(b_only), ha="center", va="center", fontsize=14)

ax.text(0.30, 0.82, "Study > 10 hours", ha="center")
ax.text(0.70, 0.82, "Attendance > 80%", ha="center")

ax.set_xlim(0, 1)
ax.set_ylim(0.15, 0.9)
ax.set_aspect("equal")
ax.axis("off")
plt.show()

## 6. Contingency Table and Probability Calculations

In [ ]:
table = pd.crosstab(
    df["group_discussion"],
    df["final_exam_pass"],
    margins=True
)

table

In [ ]:
yes_pass = ((df["group_discussion"] == "Yes") &
            (df["final_exam_pass"] == "Pass")).sum()

yes_total = (df["group_discussion"] == "Yes").sum()

joint_probability = yes_pass / total
marginal_probability = pass_count / total
conditional_probability = yes_pass / yes_total

print("Joint P(Group Discussion and Pass) =", round(joint_probability, 4))
print("Marginal P(Pass) =", round(marginal_probability, 4))
print("Conditional P(Pass | Group Discussion) =",
      round(conditional_probability, 4))

### Interpretation

Conditional probability tells us the probability of passing when we already know that the student participated in group discussion.

To check independence, compare:

**P(Pass | Group Discussion)** and **P(Pass)**.

If the values are equal, the events are independent. If they are different, the events are dependent.

Group discussion and passing are not mutually exclusive because a student can participate in group discussion and pass the exam at the same time.

In [ ]:
difference = abs(conditional_probability - marginal_probability)

print("P(Pass) =", round(marginal_probability, 4))
print("P(Pass | Group Discussion) =", round(conditional_probability, 4))

if difference < 0.01:
    print("The events are approximately independent.")
else:
    print("The events are dependent.")

## 7. Bayes Theorem

Given:

- P(High Attendance | Pass) = 0.70
- P(High Attendance | Fail) = 0.40
- P(High Attendance) = 0.60

Let H represent high attendance and P represent passing.

Using total probability:

**P(H) = P(H|P)P(P) + P(H|Fail)P(Fail)**

Solving gives:

**P(Pass) = 0.6667**

Now using Bayes theorem:

**P(Pass|H) = P(H|Pass)P(Pass) / P(H)**

Therefore:

**P(Pass|High Attendance) = 0.7778 = 77.78%**

In [ ]:
p_high_given_pass = 0.70
p_high_given_fail = 0.40
p_high = 0.60

p_pass = (p_high - p_high_given_fail) / (
    p_high_given_pass - p_high_given_fail
)

p_fail = 1 - p_pass

bayes_probability = (
    p_high_given_pass * p_pass
) / p_high

print("P(Pass) =", round(p_pass, 4))
print("P(Fail) =", round(p_fail, 4))
print("P(Pass | High Attendance) =", round(bayes_probability, 4))
print("Percentage =", round(bayes_probability * 100, 2), "%")

## 8. Final Summary

In [ ]:
summary = df.groupby("final_exam_pass")[
    ["study_hours", "attendance", "previous_test_score"]
].mean()

summary

The analysis uses study hours, attendance, group discussion participation and previous test score to understand student performance.

The probability calculations, contingency table, random variable distribution and Bayes theorem provide different ways to study the likelihood of passing the final exam.

From the given Bayes theorem information, the probability of passing given high attendance is **77.78%**.